In [24]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.multioutput import ClassifierChain
import pandas as pd
import numpy as np


In [2]:
import re

regex = re.compile(r"\[|\]|<", re.IGNORECASE)
label_path = 'simulations/data/run_1/trinucleotides_counts_sampling_0.01.csv'
label = pd.read_csv(label_path, index_col=0, header=0)
label.columns = [regex.sub("|", col) if any(x in str(col) for x in set(('[', ']', '<'))) else col for col in label.columns.values]
label.shape

(18119, 96)

In [3]:
target_path = 'simulations/ground_truth/bin_exposures.csv'
target = pd.read_csv(target_path, index_col=0)
target.shape

(18119, 29)

In [4]:
X_train, X_test, Y_train, Y_test = train_test_split(label, target, test_size=0.2, random_state=42)
bst = XGBClassifier(random_state=42)

In [5]:
from sklearn.metrics import accuracy_score
accuracies = []
for i in range(len(target.columns)):
    print(f'Training for signature {target.columns[i]}')
    bst.fit(X_train, Y_train.iloc[:, i])
    Y_pred = bst.predict(X_test)
    acc = accuracy_score(Y_test.iloc[:, i], Y_pred)
    accuracies.append(acc)
    print(f'Accuracy for signature {target.columns[i]}: {acc}')

Training for signature S1 (SBS1 - 0.99)
Accuracy for signature S1 (SBS1 - 0.99): 0.9762693156732892
Training for signature S2 (SBS2 - 0.99)
Accuracy for signature S2 (SBS2 - 0.99): 0.8752759381898455
Training for signature S3 (SBS3 - 0.97)
Accuracy for signature S3 (SBS3 - 0.97): 0.7375827814569537
Training for signature S4 (SBS4 - 0.98)
Accuracy for signature S4 (SBS4 - 0.98): 0.6760485651214128
Training for signature S5 (SBS5 - 0.98)
Accuracy for signature S5 (SBS5 - 0.98): 0.8559602649006622
Training for signature S6 (SBS7a - 1.00)
Accuracy for signature S6 (SBS7a - 1.00): 0.8137417218543046
Training for signature S7 (SBS7b - 0.96)
Accuracy for signature S7 (SBS7b - 0.96): 0.7287527593818984
Training for signature S8 (SBS8 - 0.92)
Accuracy for signature S8 (SBS8 - 0.92): 0.6763245033112583
Training for signature S9 (SBS9 - 0.94)
Accuracy for signature S9 (SBS9 - 0.94): 0.7113686534216336
Training for signature S10 (SBS10a - 1.00)
Accuracy for signature S10 (SBS10a - 1.00): 0.7248896

In [6]:
print(min(accuracies))
print(max(accuracies))
print(sum(accuracies)/len(accuracies))

0.6561810154525386
0.9762693156732892
0.7690873106493111


### Classification Chain Method

In [44]:
from sklearn.metrics import classification_report

In [ ]:
bst = XGBClassifier(random_state=42)

bst.fit(X_train, Y_train)
Y_pred = bst.predict(X_test)
print(classification_report(Y_test, Y_pred, target_names=target.columns))

                     precision    recall  f1-score   support

   S1 (SBS1 - 0.99)       0.98      0.99      0.99      3478
   S2 (SBS2 - 0.99)       0.90      0.95      0.93      3108
   S3 (SBS3 - 0.97)       0.76      0.82      0.79      2178
   S4 (SBS4 - 0.98)       0.69      0.71      0.70      2012
   S5 (SBS5 - 0.98)       0.86      0.93      0.90      2447
  S6 (SBS7a - 1.00)       0.83      0.95      0.89      2803
  S7 (SBS7b - 0.96)       0.59      0.38      0.46      1120
   S8 (SBS8 - 0.92)       0.67      0.60      0.63      1641
   S9 (SBS9 - 0.94)       0.68      0.59      0.63      1506
S10 (SBS10a - 1.00)       0.71      0.52      0.60      1379
S11 (SBS10d - 0.98)       0.60      0.50      0.54      1425
 S12 (SBS11 - 0.99)       0.40      0.13      0.20       750
 S13 (SBS13 - 0.99)       0.85      0.92      0.88      2774
 S14 (SBS14 - 0.98)       0.48      0.17      0.25       847
 S15 (SBS15 - 0.97)       0.57      0.41      0.47      1400
 S16 (SBS17 - 0.99)    

In [ ]:
chain_bst = ClassifierChain(XGBClassifier(random_state=42), order='random', random_state=42)

chain_bst.fit(X_train, Y_train)
Y_pred = chain_bst.predict(X_test)
print(classification_report(Y_test, Y_pred, target_names=target.columns))

                     precision    recall  f1-score   support

   S1 (SBS1 - 0.99)       0.98      0.99      0.99      3478
   S2 (SBS2 - 0.99)       0.90      0.95      0.93      3108
   S3 (SBS3 - 0.97)       0.76      0.84      0.80      2178
   S4 (SBS4 - 0.98)       0.66      0.77      0.71      2012
   S5 (SBS5 - 0.98)       0.88      0.91      0.90      2447
  S6 (SBS7a - 1.00)       0.83      0.93      0.88      2803
  S7 (SBS7b - 0.96)       0.59      0.32      0.41      1120
   S8 (SBS8 - 0.92)       0.65      0.59      0.62      1641
   S9 (SBS9 - 0.94)       0.67      0.61      0.64      1506
S10 (SBS10a - 1.00)       0.62      0.60      0.61      1379
S11 (SBS10d - 0.98)       0.57      0.62      0.60      1425
 S12 (SBS11 - 0.99)       0.42      0.15      0.22       750
 S13 (SBS13 - 0.99)       0.84      0.93      0.88      2774
 S14 (SBS14 - 0.98)       0.45      0.27      0.33       847
 S15 (SBS15 - 0.97)       0.56      0.44      0.49      1400
 S16 (SBS17 - 0.99)    

In [38]:
train_dataset = pd.concat([X_train, Y_train], axis=1)
test_dataset = pd.concat([X_test, Y_test], axis=1)
accuracies = {}
for i in range(len(train_dataset.columns)-1):
    bst = XGBClassifier(random_state=42)
    index = len(X_train.columns)+i
    print(f'Traning index for features begin at 0 and end at {index}')
    print(f'Training index for target is {index+1}')
    bst.fit(train_dataset.iloc[:,:index], train_dataset.iloc[:, index+1])
    Y_pred = bst.predict(test_dataset.iloc[:,:index])
    acc = accuracy_score(test_dataset.iloc[:, index+1], Y_pred)
    accuracies[f'0-{index}={index+1}'] = acc

Traning index for features begin at 0 and end at 96
Training index for target is 97
Traning index for features begin at 0 and end at 97
Training index for target is 98
Traning index for features begin at 0 and end at 98
Training index for target is 99
Traning index for features begin at 0 and end at 99
Training index for target is 100
Traning index for features begin at 0 and end at 100
Training index for target is 101
Traning index for features begin at 0 and end at 101
Training index for target is 102
Traning index for features begin at 0 and end at 102
Training index for target is 103
Traning index for features begin at 0 and end at 103
Training index for target is 104
Traning index for features begin at 0 and end at 104
Training index for target is 105
Traning index for features begin at 0 and end at 105
Training index for target is 106
Traning index for features begin at 0 and end at 106
Training index for target is 107
Traning index for features begin at 0 and end at 107
Training

IndexError: single positional indexer is out-of-bounds